# Module 1 Lab: Tracing the Northwind Apparel Agent with Phoenix

In this notebook, you will build and evaluate a database-backed AI assistant for the Northwind Apparel ecommerce catalog. 

The assistant uses a local SQLite database containing seeded sample data, including products, variants, inventory, customers, orders, shipments, carts, returns, and refunds. 

You will instrument both model and database-tool calls with Phoenix so that each interaction can be inspected for observability, debugging, and failure analysis. 

By the end of the lab, you will have a Claude Haiku agent capable of answering catalog and ecommerce-related questions using SQL-backed tools, along with traces that help you assess the agent’s behavior and identify areas for improvement. 

The workflow combines database initialization, tracing setup, tool instrumentation, agent construction, and evaluation through a batch of user queries.

## Table of Contents
- [Imports](#Imports)
- [1 - Catalog and Database Initialization](#1---catalog-and-database-initialization)
- [2 - Registering the Phoenix Tracer](#2---registering-the-phoenix-tracer)
    - **[Exercise 1 - register_tracer](#exercise-1---register_tracer)**
- [3 - Tool Wrapping and Instrumentation](#3---tool-wrapping-and-instrumentation)
- [4 - Building the Agent Infrastructure](#4---building-the-agent-infrastructure)
    - **[Exercise 2 - instrument_tool](#exercise-2---instrument_tool)**
    - **[Exercise 3 - ask_agent](#exercise-3---ask_agent)**
- [5 - Evaluating the Agent with a Query Batch](#5---evaluating-the-agent)

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, make sure you ran all of the unit tests and then click on the `Submit` button on the top right of the page.
---

<a id="imports"></a>
## Imports

This cell imports the libraries and helper modules used throughout the notebook. It suppresses nonessential warning messages to keep the output focused, then loads standard Python utilities for file paths and formatted output.

It also imports pandas for working with tabular data, Phoenix for tracing and observability, and Jupyter display tools for rendering Markdown and tables.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import functools
import json
from pathlib import Path
from pprint import pprint

import pandas as pd
import phoenix as px
import unittests
import utils
from IPython.display import Markdown, display
from phoenix.otel import register

For this course, the Northwind Apparel customer-support agent has already been implemented as a reusable Python package named `apparel_agent`. Instead of building the agent, database layer, and tools from scratch in this notebook, you will import the components needed to configure, inspect, and trace the existing application.

The package separates the application into several layers:

- `LLMAgent` imports the agent class responsible for coordinating the language model and the available tools. Later in the notebook, this class is used to create the Claude Haiku-powered agent that receives customer questions and decides when to call a tool.
- `create_seeded_connection` creates a connection to the seeded Northwind Apparel SQLite database. This database contains the catalog, inventory, customers, orders, carts, returns, refunds, and other sample ecommerce data used throughout the lab.
- `list_table_names` and `table_rows` are helper functions for exploring the database structure and previewing its contents. They help verify that the seeded data is available before the agent begins using it.
- `ApparelRepository` provides a structured interface between the agent tools and the database. Rather than having tools issue database queries directly, the repository centralizes the application’s database-backed operations.
- `ToolSpec` defines the metadata and behavior required for an agent tool, while `build_tool_registry` creates the complete set of tools available to the agent. These tools support tasks such as catalog searches, inventory checks, order lookups, tracking requests, cart creation, coupon application, returns, exchanges, and refunds.

In [2]:
from apparel_agent.agent.llm_agent import LLMAgent
from apparel_agent.data.database import (
    create_seeded_connection,
    list_table_names,
    table_rows,
)
from apparel_agent.data.repository import ApparelRepository
from apparel_agent.tools.tool_registry import ToolSpec, build_tool_registry

<a id="1---catalog-and-database-initialization"></a>
## 1 - Catalog and Database Initialization

This notebook uses a local SQLite database containing seeded Northwind Apparel catalog and ecommerce data. Because SQLite runs directly from a file and requires no separate database server, it is well suited to an interactive lab environment. The seeded database includes product information, available colors and sizes, store-level inventory, customer records, orders, shipment tracking events, and supporting tables for carts, reservations, exchanges, and refunds.

The following cell establishes the database connection used throughout the notebook. `DB_PATH` identifies the SQLite database file created for the lab, and `create_seeded_connection()` opens a connection to that file while loading the sample data required for the exercises.

Passing `reset=True` recreates the database from its seed data every time the cell runs. This provides a consistent starting point by removing changes from earlier notebook runs, including newly created carts, reservations, exchange requests, and refunds.

In [3]:
DB_PATH = Path("apparel_shop_lab.sqlite")
connection = create_seeded_connection(DB_PATH, reset=True)

`TABLE_ORDER` controls the order in which tables appear in the notebook output. The sequence begins with the product catalog and its available options, then moves through inventory, customer and order data, shipment tracking, and supporting ecommerce workflow tables.

In [4]:
TABLE_ORDER = [
    "products",
    "product_colors",
    "product_sizes",
    "inventory",
    "customers",
    "orders",
    "order_items",
    "tracking_events",
    "coupons",
    "carts",
    "cart_items",
    "reservations",
    "exchange_requests",
    "refunds",
]

`TABLE_DESCRIPTIONS` supplies a short explanation for each table. These descriptions appear above the corresponding output, making it easier to identify the purpose of the displayed records. The refunds table is expected to have no rows immediately after seeding because it is populated only when a refund is processed.

In [5]:
TABLE_DESCRIPTIONS = {
    "products": "Catalog product facts, including descriptions and care instructions.",
    "product_colors": "All color options for each product.",
    "product_sizes": "All size options for each product.",
    "inventory": "Live stock by product, size, and store.",
    "customers": "Customer profiles used for identity verification before refunds.",
    "orders": "Existing customer orders, including the 4521/4512 transposition pair and 45-day-old 4530.",
    "order_items": "Line items attached to each order.",
    "tracking_events": "Timestamped shipment updates for orders.",
    "coupons": "Available coupon codes and discount rates.",
    "carts": "Carts created while the agent is running.",
    "cart_items": "Items added to carts while the agent is running.",
    "reservations": "Pickup reservations created while the agent is running.",
    "exchange_requests": "Exchange requests created while the agent is running.",
    "refunds": "Refund records written by process_refund. Empty at seed time.",
}

`display_tables()` checks which tables exist in the database and loops through them using the predefined display order. For each table, it retrieves the rows, displays the table name and row count, and shows its description.

Tables containing data are rendered as pandas DataFrames. Empty tables display a short placeholder message instead. Calling `display_tables()` at the end of the cell produces a compact, readable inventory of the seeded database.

In [6]:
def display_tables():
    existing_tables = set(list_table_names(connection))
    for table_name in TABLE_ORDER:
        if table_name not in existing_tables:
            continue
        rows = table_rows(connection, table_name)
        display(
            Markdown(
                f"### `{table_name}` ({len(rows)} rows)\n{TABLE_DESCRIPTIONS[table_name]}"
            )
        )
        if rows:
            display(pd.DataFrame(rows))
        else:
            display(Markdown("_No rows yet._"))

display_tables()

### `products` (20 rows)
Catalog product facts, including descriptions and care instructions.

,id,name,category,color,fabric,description,price,available,returnable,final_sale,care
0,TEE-001,Classic Tee,tops,white,combed cotton,100% combed cotton crew-neck t-shirt. Pre-shru...,24.99,1,1,0,
1,SHO-001,Trail Runner,shoes,black,synthetic mesh,Lightweight trail running shoe with rock plate...,89.99,1,1,0,
2,SHO-002,Court Shoe,shoes,white,canvas and rubber,Low-profile court shoe with gum rubber outsole...,74.99,1,1,0,
3,SHO-003,Slip-On Canvas,shoes,navy,canvas,Canvas slip-on shoe with elastic side panels. ...,55.00,0,1,0,
4,OUT-001,Denim Jacket,outerwear,blue,cotton denim,Classic trucker-style denim jacket. Button fro...,98.00,1,1,0,
5,OUT-002,Raincoat,outerwear,yellow,waterproof nylon,Waterproof hooded raincoat with sealed seams a...,125.00,1,1,0,
6,OUT-003,Fleece Pullover,outerwear,heather gray,polyester fleece,Quarter-zip midweight fleece pullover. Chest p...,68.00,1,1,0,
7,TOP-002,Oxford Shirt,tops,white,Oxford cotton,Button-down Oxford cloth shirt with chest pock...,68.00,1,1,0,
8,TOP-003,Hooded Sweatshirt,tops,gray,cotton blend,Midweight cotton-blend hoodie with kangaroo po...,54.00,1,1,0,
9,TOP-004,Crew Neck Sweater,tops,cream,merino wool,Merino wool crew-neck sweater. Machine washabl...,72.00,1,1,0,Machine washable on wool cycle.


### `product_colors` (54 rows)
All color options for each product.

,product_id,color
0,TEE-001,white
1,TEE-001,black
2,TEE-001,navy
3,TEE-001,heather gray
4,SHO-001,black
5,SHO-001,gray
6,SHO-002,white
7,SHO-002,black
8,SHO-003,navy
9,SHO-003,gray


### `product_sizes` (72 rows)
All size options for each product.

,product_id,size
0,TEE-001,XS
1,TEE-001,S
2,TEE-001,M
3,TEE-001,L
4,TEE-001,XL
...,...,...
67,ACC-004,one size
68,ACC-005,M
69,ACC-005,L
70,ACC-006,M


... (18 outputs left out)

### `exchange_requests` (0 rows)
Exchange requests created while the agent is running.

_No rows yet._

### `refunds` (0 rows)
Refund records written by process_refund. Empty at seed time.

_No rows yet._

<a id="2---registering-the-phoenix-tracer"></a>
## 2 - Registering the Phoenix Tracer

This cell launches the Phoenix application, which provides a local interface for tracing and inspecting the agent's activity. Phoenix can be used to review model calls, tool usage, inputs, outputs, and other observability data generated during the notebook exercises.

**Note: You will get a warning about aioboto3 not being installed. This library is not required for this or any of the graded labs so feel free to dismiss it.**

In [7]:
# Launch Phoenix
px.launch_app()

boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


When Phoenix starts, it may display `http://localhost:6006/` as the application URL. Although this address works when Phoenix is running on your local computer, this notebook is executed in a hosted virtual environment. Therefore, `localhost` refers to the remote notebook environment rather than your own browser.

Run the next cell to generate the correct browser link for accessing the Phoenix interface in a new tab. For the remaining exercises, continue using `http://localhost:6006` as the tracing endpoint, since Phoenix is running locally within the notebook environment.

In [8]:
# Show the true Phoenix URL
utils.show_phoenix_link()

<a id='exercise-1---register_tracer'></a>
#### Exercise 1 - register_tracer

**Your Task:**

Register a Phoenix tracer so supported Anthropic model calls in this notebook can be traced automatically. Complete the function in two steps:

* **Register the tracer provider**:
    * Call `register` with the project name `"dlai-ecommerce-agent"`, set `auto_instrument=True` so supported libraries are wrapped automatically, and point `endpoint` to the local collector at `"http://localhost:6006/v1/traces"`.

* **Get a tracer**:
    * Use the returned `tracer_provider` to obtain a tracer named `"ecommerce-agent"` via its `get_tracer` method.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

* For `tracer_provider`: Call `register(project_name="dlai-ecommerce-agent", auto_instrument=True, endpoint="http://localhost:6006/v1/traces")`.
* For `tracer`: Call `tracer_provider.get_tracer("ecommerce-agent")`.

</details>

In [9]:
# GRADED FUNCTION: register_tracer

def register_tracer():
    
    ### START CODE HERE ###
    
    tracer_provider = register(project_name="dlai-ecommerce-agent", auto_instrument=True, endpoint="http://localhost:6006/v1/traces")
    
    tracer = tracer_provider.get_tracer("ecommerce-agent")
    
    ### END CODE HERE ###

    return tracer

In [10]:
tracer = register_tracer()

# Confirm the tracer can open a span without error
try:
    with tracer.start_as_current_span("connection-check"):
        print("Tracer is ready and can record spans.")
except Exception as e:
    print(f"There was an issue with the tracer. Details: {e}")

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dlai-ecommerce-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Tracer is ready and can record spans.


**Expected Output**

```text
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dlai-ecommerce-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Tracer is ready and can record spans.```

In [11]:
# Test your code!
unittests.test_register_tracer(tracer)

 All tests passed!


<a id="3---tool-wrapping-and-instrumentation"></a>
## 3 - Tool Wrapping and Instrumentation

This cell creates an `ApparelRepository` connected to the seeded database. The repository provides a structured interface for reading and updating ecommerce data without requiring the agent to write raw SQL directly.

Next, `build_tool_registry(repo)` creates the set of tools available to the agent, using the repository to perform database-backed actions such as searching products, checking inventory, reviewing orders, or processing supported workflows.

The final two lines confirm the setup by printing the number of registered tools and a sorted list of their names.

In [12]:
repo = ApparelRepository(connection=connection)
tool_registry = build_tool_registry(repo)

print(f"Connected {len(tool_registry)} tools:")
print(", ".join(sorted(tool_registry)))

Connected 13 tools:
apply_coupon, check_inventory, check_return_policy, create_cart, create_exchange_request, get_customer, get_order_tracking, get_product_details, lookup_order, order_lookup, process_refund, reserve_item, search_catalog


This cell defines a small set of example tool calls and runs each one against the registered tool registry. Each example consists of a tool name and the keyword arguments required by that tool.

The examples demonstrate several core agent capabilities:

- Searching the catalog for a specific product or product category.
- Checking live inventory for a product and size.
- Retrieving detailed product information, including catalog-provided details such as care instructions.

The loop retrieves each tool from `tool_registry`, calls its underlying function with the provided arguments, and uses `pprint()` to display the returned result in a readable format. Running these examples verifies that the tools are registered correctly and can access the seeded database.

In [13]:
capability_examples = [
    ("search_catalog", {"query": "Classic Tee", "filters": {}}),
    ("check_inventory", {"product_id": "TEE-001", "size": "M"}),
    ("get_product_details", {"product_id": "TOP-004"}),
    ("search_catalog", {"query": "shoes", "filters": {"category": "shoes"}}),
    ("check_inventory", {"product_id": "SHO-002", "size": "12"}),
]

for tool_name, args in capability_examples:
    print(f"\nTOOL: {tool_name}")
    pprint(tool_registry[tool_name].func(**args))


TOOL: search_catalog
{'count': 3,
 'filters': {},
 'products': [{'available': True,
               'care': '',
               'category': 'tops',
               'color': 'white',
               'colors': ['black', 'heather gray', 'navy', 'white'],
               'description': '100% combed cotton crew-neck t-shirt. '
                              'Pre-shrunk, soft hand feel.',
               'fabric': 'combed cotton',
               'final_sale': False,
               'id': 'TEE-001',
               'name': 'Classic Tee',
               'price': 24.99,
               'returnable': True,
               'sizes': ['L', 'M', 'S', 'XL', 'XS']},
              {'available': True,
               'care': '',
               'category': 'tops',
               'color': 'white',
               'colors': ['blue', 'pink', 'white'],
               'description': 'Button-down Oxford cloth shirt with chest '
                              'pocket. Classic fit.',
               'fabric': 'Oxford cotton',

<a id='4---building-the-agent-infrastructure'></a>
## 4 - Building the Agent Infrastructure

In this section, you will add tracing to the apparel agent’s tool calls and create a helper function for sending questions to the agent. 

The `instrument_tool` function should wrap an individual tool so that Phoenix records each invocation as a trace span. Each span should capture useful information about the tool call, including the tool name, the arguments provided by the agent, the tool’s response, and any errors that occur.

After the tools are instrumented, the agent will use the traced versions in its tool registry. This allows you to inspect how the agent selects tools, which inputs it sends to them, and whether the returned data supports the final response.

The `ask_agent` function should provide a simple interface for submitting a user question to the `LLMAgent`. It should call the agent, return its final response, and create a trace for the overall request.

When you run `ask_agent`, Phoenix should show the complete execution path for that interaction, including the model call, any tool calls the model decides to make, the arguments passed to each tool, and the final answer returned to the user. This makes it easier to evaluate whether the agent used the appropriate data source and to diagnose unexpected behavior.

The agent is configured with Claude Haiku and a temperature of `0.0` to produce more consistent responses.

<a id='exercise-2---instrument_tool'></a>
### Exercise 2 - instrument_tool

**Your Task:**

Complete `instrument_tool` to create and return a traced copy of a tool specification using OpenTelemetry/OpenInference spans.

1. Define a wrapper function `traced_func` around `spec.func` using `@functools.wraps(spec.func)` that accepts `*args` and `**kwargs`.
2. Inside `traced_func`, use `tracer.start_as_current_span(spec.name)` in a context manager to track execution.
3. Set the required OpenInference attributes on the span:
   - Set `"openinference.span.kind"` to `"TOOL"`.
   - Set `"tool.name"` to `spec.name`.
   - Serialize `kwargs` to JSON with `json.dumps(kwargs, default=str)` and set it as `"input.value"`.
   - Set `"input.mime_type"` to `"application/json"`.
4. Execute the original tool function `spec.func(*args, **kwargs)` and store the result.
5. Serialize the execution result to JSON and set it as `"output.value"`, along with setting `"output.mime_type"` to `"application/json"`.
6. Return the execution result from the wrapper.
7. Create and return a new `ToolSpec` copying `name`, `description`, and `input_schema` from `spec`, assigning `func=traced_func`.

<details>
  <summary><b><font color="green">Additional Code Hints & Solution (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's the complete implementation:

```python
    @functools.wraps(spec.func)
    def traced_func(*args, **kwargs):
        with tracer.start_as_current_span(spec.name) as span:
            span.set_attribute("openinference.span.kind", "TOOL")
            span.set_attribute("tool.name", spec.name)
            span.set_attribute("input.value", json.dumps(kwargs, default=str))
            span.set_attribute("input.mime_type", "application/json")
            
            result = spec.func(*args, **kwargs)
            
            span.set_attribute("output.value", json.dumps(result, default=str))
            span.set_attribute("output.mime_type", "application/json")
            return result
    
    tool_spec = ToolSpec(
        name=spec.name,
        description=spec.description,
        input_schema=spec.input_schema,
        func=traced_func,
    )
```

</details>

In [14]:
# GRADED FUNCTION: instrument_tool

def instrument_tool(tracer, spec):
    
    ### START CODE HERE ###
    
    # Use the functools.wraps decorator to preserve the metadata of original function
    @functools.wraps(spec.func)
    # Define the correct name of the function with the correct params (*args, **kwargs)
    def traced_func(*args, **kwargs):
        # Enable tracing of spans by using the context manager
        with tracer.start_as_current_span(spec.name) as span:

            # Set attributes of span (already provided)
            span.set_attribute("openinference.span.kind", "TOOL") 
            span.set_attribute("tool.name", spec.name) 
            span.set_attribute("input.value", json.dumps(kwargs, default=str)) 
            span.set_attribute("input.mime_type", "application/json") 

            # Run original function and save result
            result = spec.func(*args, **kwargs)

            # Set attributes of span (already provided)
            span.set_attribute("output.value", json.dumps(result, default=str)) 
            span.set_attribute("output.mime_type", "application/json") 
            return result
    
    tool_spec = ToolSpec(
        name=spec.name,
        description=spec.description,
        input_schema=spec.input_schema,
        func=traced_func,
    ) 
    
    ### END CODE HERE ###

    return tool_spec

This cell creates a traced copy of every tool in `tool_registry` by applying `instrument_tool()` to each tool specification.

It then runs a traced `search_catalog` call for `"Classic Tee"`. The result is unchanged, but the call is also recorded as a Phoenix span. The final print statements confirm the number of traced tools and display the top search result.

In [15]:
# Instrument every tool in the registry
traced_tool_registry = {name: instrument_tool(tracer, spec) for name, spec in tool_registry.items()}

# Try a traced tool — it returns the same result as the untraced tool, but now records a span
traced_result = traced_tool_registry["search_catalog"].func(
    query="Classic Tee",
    filters={},
)
print("Traced tools:", len(traced_tool_registry))
print("Top match:", traced_result["products"][0]["id"], "-", traced_result["products"][0]["name"])

Traced tools: 13
Top match: TEE-001 - Classic Tee


**Expected Output**

```text
Traced tools: 13
Top match: TEE-001 - Classic Tee
```

In [16]:
# Test your code!
unittests.test_instrument_tool(traced_tool_registry)

 All tests passed!


This cell creates an `LLMAgent` with access to the traced tool registry. The agent uses the specified Claude Haiku model with a temperature of `0.0` for consistent, deterministic responses.

Setting `weak_prompt_mode=False` enables the agent’s full prompt configuration. The final message confirms that the apparel agent is ready to use.

In [17]:
agent = LLMAgent(
    traced_tool_registry,
    model_name="claude-haiku-4-5-20251001",
    temperature=0.0,
    weak_prompt_mode=False,
)

print("apparel_agent ready!")

apparel_agent ready!


<a id='exercise-3---ask_agent'></a>
### Exercise 3 - ask_agent

**Your Task:**

Instrument the agent-call function so the complete agent execution is captured as a Phoenix `chain` span.

Inside `ask_agent`:

* **Trace the function** with the `tracer.chain` decorator:
    * Use `@tracer.chain` above `ask_agent` so each invocation opens a `chain` span.

* **Run the agent** with the provided user query:
    * Call `agent.run(user_query)` and store its response in `result`.

* **Return the agent response**:
    * Return `result` from the function.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

1. Add `@tracer.chain` above the function to trace the complete agent workflow.
2. Call `agent.run(user_query)` and store the response in `result`.
3. Return `result` so the final agent response is available to the next cell.

```python
@tracer.chain
def ask_agent(agent, user_query):
    result = agent.run(user_query)
    return result
```
</details>

In [18]:
# GRADED FUNCTION: ask_agent

### START CODE HERE ###

# Use the correct decorator to trace the function
@tracer.chain
def ask_agent(agent, user_query): 

    # Use the agent's run method and pass the user query 
    result = agent.run(user_query)

    # Return the result
    return result
    
### END CODE HERE ###

This cell sends the apparel agent a product question it should be able to answer using the catalog tools. `ask_agent()` runs the agent and returns its final response along with details about any tool calls made during the interaction.

The cell then prints the agent’s answer and the names of the tools it used, making it possible to verify both the response and the tool-selection behavior.

In [19]:
# Try the agent on a question it should handle well
result = ask_agent(agent, "What colors does the Classic Tee come in?")
print("ANSWER:", result.final_answer)
print("TOOLS USED:", [call.name for call in result.tool_calls])

ANSWER: The Classic Tee comes in white, black, navy, and heather gray.
TOOLS USED: ['search_catalog']


**Expected Output**

```text
ANSWER: The Classic Tee comes in white, black, navy, and heather gray.
TOOLS USED: ['search_catalog']
```

In [20]:
# Test your code!
unittests.test_ask_agent(ask_agent, traced_tool_registry)

 All tests passed!


<a id='5---evaluating-the-agent'></a>
## 5 - Evaluating the Agent with a Query Batch

The following cell runs the agent against a varied set of customer queries, including straightforward catalog lookups, multi-part questions, repeated open-ended prompts, vague shopping requests, product-care questions, unsupported requests, and more complex outfit or inventory needs. For each query, it calls `ask_agent()`, then prints the question, the first 400 characters of the response, and the names of any tools used. Reviewing the output helps assess whether the agent provides accurate catalog-grounded answers, uses tools appropriately, behaves consistently across repeated prompts, handles ambiguity helpfully, and avoids inventing information or responding beyond its supported scope.

In [21]:
test_queries = [
    # Baseline catalog questions
    "How much is the Wool Beanie?",
    "Do you sell raincoats?",
    "I need something to keep my head warm in winter.",

    # Multi-part
    "Can I get the Classic Tee in medium, and what's your return policy?",

    # Same open-ended question, repeated three times
    "What's a good gift idea for someone who works from home?",
    "What's a good gift idea for someone who works from home?",
    "What's a good gift idea for someone who works from home?",

    # Vague prompts
    "I need something comfortable for a long flight.",
    "What should I buy for a casual weekend?",

    # Faithfulness drift bait
    "How should I care for the merino wool sweater?",

    # Scope/refusal failures
    "Where's my order #4521?",
    "What's your phone number for customer service?",

    # Complex prompts that should pass
    "Can you help me put together a winter outfit with a sweater, pants, and a beanie?",
    "What's the cheapest pair of shoes you have, and does it come in size 12?",
]

for q in test_queries:
    result = ask_agent(agent, q)
    print(f"\nQ: {q}")
    print(f"A: {result.final_answer[:400]}")
    print("TOOLS:", [call.name for call in result.tool_calls])
    print("-" * 60)


Q: How much is the Wool Beanie?
A: The Wool Beanie is $22.00.
TOOLS: ['search_catalog']
------------------------------------------------------------

Q: Do you sell raincoats?
A: Yes. Northwind Apparel sells the Raincoat for $125.00. It comes in yellow and navy, with sizes S, M, and L.
TOOLS: ['search_catalog']
------------------------------------------------------------

Q: I need something to keep my head warm in winter.
A: The Wool Beanie is the best match. It is a cuffed merino-wool beanie, one size fits most, and costs $22.00.
TOOLS: ['search_catalog']
------------------------------------------------------------

Q: Can I get the Classic Tee in medium, and what's your return policy?
A: Yes, the Classic Tee is available in size M. I do not have Northwind Apparel's full general return policy in the catalog data I can see, so I should not invent one.
TOOLS: ['search_catalog', 'check_inventory']
------------------------------------------------------------

Q: What's a good gift idea 

**Review Your Traces in Phoenix**

Run the next cell to open the correct Phoenix interface for this hosted notebook environment.

In Phoenix, inspect the recorded chain, model, and tool spans. Pay particular attention to:

- The user query and the agent’s final response.
- Which tools the agent selected for each request.
- The arguments sent to each tool and the returned results.
- Whether the agent used the available data correctly.
- Any errors, unexpected tool calls, or incomplete responses.

In [22]:
utils.show_phoenix_link()

**Conclusion**

This notebook demonstrated how to create a database-backed ecommerce agent while maintaining visibility into its behavior through end-to-end tracing. By connecting the Northwind Apparel SQLite dataset to tool-enabled agent workflows, the assistant can retrieve grounded information about products, inventory, customers, and orders instead of relying only on model-generated responses. Phoenix instrumentation makes it possible to inspect model and tool calls, diagnose errors, and evaluate failure modes across a batch of queries. This approach provides a practical foundation for building more reliable AI assistants, and it can be extended with stronger validation, improved SQL safeguards, additional tools, and more comprehensive evaluation datasets